In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torchvision.transforms as T

class Transformer(nn.Module):
    def __init__(self, input_dim, model_dim, num_heads, num_layers, dim_feedforward, num_classes, seq_len):
        super(Transformer, self).__init__()
        self.model_dim = model_dim
        self.embedding = nn.Linear(input_dim, model_dim)  # Embed input pixels
        self.positional_encoding = nn.Parameter(torch.zeros(1, seq_len, model_dim))  # Learnable positional encoding
        
        # Define transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim, 
            nhead=num_heads, 
            dim_feedforward=dim_feedforward, 
            dropout=0.1,
            activation='relu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Linear(model_dim, num_classes)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()  # x: (batch_size, seq_len, input_dim)
        
        x = self.embedding(x)  # (batch_size, seq_len, model_dim)
        x = x + self.positional_encoding[:, :seq_len, :]  # Add positional encoding
        
        # Transformer encoder expects (seq_len, batch_size, model_dim)
        x = x.permute(1, 0, 2)  # (seq_len, batch_size, model_dim)
        x = self.transformer_encoder(x)  # (seq_len, batch_size, model_dim)
        
        # Use the representation of the last time step for classification
        x = x[-1, :, :]  # (batch_size, model_dim)
        x = self.classifier(x)  # (batch_size, num_classes)
        
        return x

# Updated hyperparameters
input_dim = 1
model_dim = 128  # hidden_size
num_heads = 4  # num_heads
num_layers = 2  # num_hidden_layers
dim_feedforward = 512  # filter_size
num_classes = 10
seq_len = 28 * 28  # MNIST images are 28x28 pixels
batch_size = 64
learning_rate = 1e-3
epochs = 50

# Data Preparation
transform = transforms.Compose([
    T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))]),
])
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Model, Loss, Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Transformer(input_dim, model_dim, num_heads, num_layers, dim_feedforward, num_classes, seq_len).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training Loop
for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        # flattens
        # turn [64, 28, 28] to [64, 784, 1]
        x = x.view(x.shape[0], -1, input_dim)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += (pred == y).sum().item()
    
    train_loss /= len(train_loader.dataset)
    accuracy = correct / len(train_loader.dataset)
    print(f"Epoch {epoch+1}, Loss: {train_loss:.4f}, Accuracy: {accuracy:.4f}")

# Evaluation
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        # flattens
        # turn [64, 28, 28] to [64, 784, 1]
        x = x.view(x.shape[0], -1, input_dim)
        output = model(x)
        test_loss += criterion(output, y).item()
        pred = output.argmax(dim=1)
        correct += (pred == y).sum().item()

test_loss /= len(test_loader.dataset)
accuracy = correct / len(test_loader.dataset)
print(f"Test Loss: {test_loss:.4f}, Accuracy: {accuracy:.4f}")


Epoch 1, Loss: 0.0216, Accuracy: 0.4973
Epoch 2, Loss: 0.0085, Accuracy: 0.8267
Epoch 3, Loss: 0.0055, Accuracy: 0.8897
Epoch 4, Loss: 0.0044, Accuracy: 0.9114
Epoch 5, Loss: 0.0040, Accuracy: 0.9194


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn

class Transformer(nn.Module):
    def __init__(self, input_size, model_dim, num_heads, num_layers, dim_feedforward, output_size, seq_len):
        """
        input_size: Length of x_t (e.g., 183, for x-coordinates of bounding boxes)
        model_dim: Dimensionality of the Transformer embeddings
        num_heads: Number of attention heads
        num_layers: Number of Transformer encoder layers
        dim_feedforward: Dimensionality of the feedforward layers in the Transformer
        seq_len: Maximum number of timesteps to process
        """
        super(Transformer, self).__init__()
        
        self.model_dim = model_dim
        self.input_size = input_size
        self.seq_len = seq_len
        
        # Positional encoding (learnable for handling temporal order)
        self.positional_encoding = nn.Parameter(torch.zeros(1, seq_len, model_dim))
        
        # Embed input vector into model_dim
        self.embedding = nn.Linear(input_size, model_dim)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=0.1,
            activation='relu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Output layer to map back to input_size (e.g., 183)
        self.output_layer = nn.Linear(model_dim, output_size)

    def forward(self, x_t, past_context):
        """
        x_t: (batch_size, input_size) -> Input vector at current timestep t
        past_context: (batch_size, seq_len, model_dim) -> Context from previous timesteps
        
        Returns:
        refined_prediction: (batch_size, input_size) -> Refined bounding box coordinates
        updated_context: (batch_size, seq_len, model_dim) -> Updated context including current timestep
        """
        batch_size = x_t.size(0)
        channel = xs.shape[1]
        scale = xs.shape[2]
        # Embed the current input
        x_t_emb = self.embedding(x_t)  # (batch_size, model_dim)
        x_t_emb = x_t_emb.unsqueeze(1)  # Add sequence dimension: (batch_size, 1, model_dim)
        
        # Combine with past context
        if past_context is None:
            # Initialize context with only the current timestep
            context = x_t_emb  # (batch_size, 1, model_dim)
        else:
            # Append current timestep to past context
            context = torch.cat([past_context, x_t_emb], dim=1)  # (batch_size, seq_len + 1, model_dim)
        
        # Handle maximum sequence length (sliding window)
        if context.size(1) > self.seq_len:
            context = context[:, -self.seq_len:, :]  # Keep only the last seq_len timesteps
        
        # Add positional encoding
        seq_len = context.size(1)
        pos_enc = self.positional_encoding[:, :seq_len, :]  # (1, seq_len, model_dim)
        context = context + pos_enc  # (batch_size, seq_len, model_dim)
        
        # Transformer expects input shape: (seq_len, batch_size, model_dim)
        context = context.permute(1, 0, 2)  # (seq_len, batch_size, model_dim)
        encoded_context = self.transformer_encoder(context)  # (seq_len, batch_size, model_dim)
        encoded_context = encoded_context.permute(1, 0, 2)  # Back to (batch_size, seq_len, model_dim)
        
        # Use the representation of the last timestep to make predictions
        current_context = encoded_context[:, -1, :]  # (batch_size, model_dim)
        refined_prediction = self.output_layer(current_context)  # (batch_size, input_size)
        
        return refined_prediction, encoded_context



In [22]:
# Initialize the model
model = Transformer(
    input_size=1083,  # Length of x_t
    model_dim=1024,   # Transformer model dimension
    num_heads=4, 
    num_layers=2, 
    dim_feedforward=512, 
    output_size = 1083,
    seq_len=8       # Max timesteps to remember
)

# Initial setup
batch_size = 32
timesteps = 10
input_size = 1083

past_context = None  # No context at the beginning

# Simulate timesteps
for t in range(timesteps):
    x_t = torch.rand(batch_size, input_size)  # Input vector at timestep t
    refined_pred, past_context = model(x_t, past_context)  # Process with Transformer
    print(f"Timestep {t}, Refined Predictions: {refined_pred.shape}")


Timestep 0, Refined Predictions: torch.Size([32, 1083])
Timestep 1, Refined Predictions: torch.Size([32, 1083])
Timestep 2, Refined Predictions: torch.Size([32, 1083])
Timestep 3, Refined Predictions: torch.Size([32, 1083])
Timestep 4, Refined Predictions: torch.Size([32, 1083])
Timestep 5, Refined Predictions: torch.Size([32, 1083])
Timestep 6, Refined Predictions: torch.Size([32, 1083])
Timestep 7, Refined Predictions: torch.Size([32, 1083])
Timestep 8, Refined Predictions: torch.Size([32, 1083])
Timestep 9, Refined Predictions: torch.Size([32, 1083])
